<a href="https://colab.research.google.com/github/ivang019/Dominance-in-Internet-Market-in-Mexico/blob/main/WC26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# BLOQUE 1 — DATOS
# Inputs : ninguno (descarga desde Kaggle)
# Outputs: resultados, elo, elo_serie, elo_pre, equipos48, alias
# Guarda : elo_pre_48.csv en Drive
# =====================================================================
!pip -q install kagglehub statsmodels
import os, json, datetime as dt, numpy as np, pandas as pd, kagglehub
pd.set_option("display.max_columns", None)
HOY = dt.date.today().isoformat()

# --- Drive -----------------------------------------------------------
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/wc2026_forecasting/v2"
os.makedirs(OUT, exist_ok=True)
print(f"Carpeta de salida: {OUT}")

# --- Descarga --------------------------------------------------------
ruta_res  = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
ruta_elo1 = kagglehub.dataset_download("afonsofernandescruz/2026-fifa-world-cup-historical-elo-ratings")
ruta_elo2 = kagglehub.dataset_download("saifalnimri/international-football-elo-ratings")

# --- Fuente A: resultados históricos ---------------------------------
resultados = pd.read_csv(os.path.join(ruta_res, "results.csv"), parse_dates=["date"])
print(f"\n[A] resultados: {resultados.shape} | max fecha: {resultados['date'].max().date()}")
assert resultados["date"].max().year >= 2025

# --- Fuente B: Elo pre-torneo de las 48 (bundle WC2026) --------------
csv_e1 = [f for f in os.listdir(ruta_elo1) if f.endswith(".csv")][0]
elo = pd.read_csv(os.path.join(ruta_elo1, csv_e1))
elo["snapshot_date"] = pd.to_datetime(elo["snapshot_date"])
print(f"[B] elo bundle:  {elo.shape}")

# --- Fuente C: Elo histórico serie temporal (saifalnimri) ------------
elo2_raw = pd.read_csv(os.path.join(ruta_elo2, "eloratings.csv"))
elo2_raw["date"] = pd.to_datetime(elo2_raw["date"], format="mixed", errors="coerce")
elo2_raw["team"] = elo2_raw["team"].str.replace("\xa0", " ", regex=False).str.strip()
elo_serie = (elo2_raw[["date","team","rating"]]
             .dropna()
             .sort_values("date")
             .reset_index(drop=True))
print(f"[C] elo_serie:   {elo_serie.shape} | equipos: {elo_serie['team'].nunique()} | "
      f"rango: {elo_serie['date'].min().date()} → {elo_serie['date'].max().date()}")

# --- elo_pre: snapshot pre-torneo de las 48 --------------------------
CUTOFF = pd.Timestamp("2026-06-10")
pre = (elo[elo["snapshot_date"] <= CUTOFF]
          .sort_values("snapshot_date")
          .groupby("country", as_index=False).tail(1).copy())

# Alias: Czechia → Czech Republic (curaduría/supuesto documentado)
cz = [c for c in pre["country"].unique() if c.lower().startswith("cze")]
alias = {cz[0]: "Czech Republic"} if cz else {}

pre["country"] = pre["country"].replace(alias)
elo_pre = (pre[pre["country"].isin(
               sorted(set(
                   resultados[(resultados["tournament"]=="FIFA World Cup") &
                              (resultados["date"].dt.year==2026)]["home_team"]) |
                   set(resultados[(resultados["tournament"]=="FIFA World Cup") &
                                  (resultados["date"].dt.year==2026)]["away_team"])
               ))]
           [["country","rating","rank","confederation","snapshot_date"]]
           .sort_values("rating", ascending=False)
           .reset_index(drop=True))

# equipos48: lista canónica de nombres (ortografía de resultados.csv)
wc26 = resultados[(resultados["tournament"]=="FIFA World Cup") &
                  (resultados["date"].dt.year==2026)]
equipos48 = sorted(set(wc26["home_team"]) | set(wc26["away_team"]))

# Verificaciones
assert len(elo_pre) == 48 and elo_pre["country"].nunique() == 48
assert len(equipos48) == 48
assert elo_pre["snapshot_date"].dt.date.unique().tolist() == [pd.Timestamp("2026-05-27").date()]
faltan = set(equipos48) - set(elo_pre["country"])
assert not faltan, f"Sin Elo pre-torneo: {faltan}"

print(f"\n[elo_pre] {len(elo_pre)} equipos | snapshot: {elo_pre['snapshot_date'].iloc[0].date()}")
print(f"  Top-5: {elo_pre[['country','rating']].head(5).to_string(index=False)}")
print(f"  Bot-5: {elo_pre[['country','rating']].tail(5).to_string(index=False)}")
print(f"[alias] {alias}")

# --- Guardar elo_pre_48 en Drive (no cambia entre versiones) ---------
elo_pre.to_csv(f"{OUT}/elo_pre_48.csv", index=False)
kb = os.path.getsize(f"{OUT}/elo_pre_48.csv") / 1024
print(f"\nGuardado: elo_pre_48.csv ({kb:.1f} KB)")

print("\n✓ BLOQUE 1 completo")
print(f"  Variables en sesión: resultados, elo, elo_serie, elo_pre, equipos48, alias")

Mounted at /content/drive
Carpeta de salida: /content/drive/MyDrive/wc2026_forecasting/v2
Using Colab cache for faster access to the 'international-football-results-from-1872-to-2017' dataset.


100%|██████████| 127k/127k [00:00<00:00, 49.1MB/s]

Extracting files...


100%|██████████| 45.6k/45.6k [00:00<00:00, 20.8MB/s]

Extracting files...

[A] resultados: (49477, 9) | max fecha: 2026-06-27
[B] elo bundle:  (4683, 23)


[C] elo_serie:   (6647, 3) | equipos: 270 | rango: 1872-11-30 → 2025-12-13

[elo_pre] 48 equipos | snapshot: 2026-05-27
  Top-5:   country  rating
    Spain    2165
Argentina    2113
   France    2081
  England    2020
   Brazil    1984
  Bot-5:      country  rating
       Haiti    1532
South Africa    1524
       Ghana    1503
     Curaçao    1436
       Qatar    1425
[alias] {'Czechia': 'Czech Republic'}

Guardado: elo_pre_48.csv (1.6 KB)

✓ BLOQUE 1 completo
  Variables en sesión: resultados, elo, elo_serie, elo_pre, equipos48, alias


In [ ]:
# =====================================================================
# ESTRUCTURA DE CARPETAS EN DRIVE
# Corre esto UNA VEZ antes de los Bloques 2-4
# =====================================================================
import os

BASE = "/content/drive/MyDrive/wc2026_forecasting/v2"

RUTAS = {
    "datos":        f"{BASE}/datos",
    "modelo":       f"{BASE}/modelo",
    "simulaciones": f"{BASE}/simulaciones",
}

for nombre, ruta in RUTAS.items():
    os.makedirs(ruta, exist_ok=True)
    print(f"  ✓ {nombre:<15} → {ruta}")

# Mover elo_pre_48.csv al lugar correcto si ya existe en v2/
origen  = f"{BASE}/elo_pre_48.csv"
destino = f"{RUTAS['datos']}/elo_pre_48.csv"
if os.path.exists(origen) and not os.path.exists(destino):
    import shutil
    shutil.move(origen, destino)
    print(f"\n  Movido: elo_pre_48.csv → datos/")
elif os.path.exists(destino):
    print(f"\n  elo_pre_48.csv ya está en datos/")

print("\n✓ Estructura lista")
print(f"  OUT_DATOS      = {RUTAS['datos']}")
print(f"  OUT_MODELO     = {RUTAS['modelo']}")
print(f"  OUT_SIMS       = {RUTAS['simulaciones']}")

  ✓ datos           → /content/drive/MyDrive/wc2026_forecasting/v2/datos
  ✓ modelo          → /content/drive/MyDrive/wc2026_forecasting/v2/modelo
  ✓ simulaciones    → /content/drive/MyDrive/wc2026_forecasting/v2/simulaciones

  elo_pre_48.csv ya está en datos/

✓ Estructura lista
  OUT_DATOS      = /content/drive/MyDrive/wc2026_forecasting/v2/datos
  OUT_MODELO     = /content/drive/MyDrive/wc2026_forecasting/v2/modelo
  OUT_SIMS       = /content/drive/MyDrive/wc2026_forecasting/v2/simulaciones


In [ ]:
# =====================================================================
# BLOQUE 2 v2 — MODELO NEUTRAL (corregido: ventana hasta 2026-06-10)
# Inputs : resultados, elo_serie (Bloque 1)
# Outputs: m_neutral, long_neutral, model_neutral
# Guarda : m_neutral_2010_2026jun.csv, long_neutral.csv,
#          metadata_modelo_neutral.json
# Decisiones:
#   - Solo partidos neutrales (neutral==True)
#   - Ventana 2010 → 2026-06-09 (día antes del torneo)
#   - Tope merge_asof: 18 meses
#   - Fórmula: goals ~ elo_diff (sin is_home)
# =====================================================================
import pandas as pd, numpy as np, warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import poisson
warnings.filterwarnings("ignore")

for v in ["resultados","elo_serie","elo_pre","equipos48","alias"]:
    assert v in dir(), f"Falta '{v}': re-corre el Bloque 1"

TOPE   = pd.Timedelta(days=548)
WIN    = pd.Timestamp("2010-01-01")
CUTOFF = pd.Timestamp("2026-06-10")   # día antes del torneo

# --- Paso 1: partidos neutrales jugados, ventana 2010→2026-06-09 ----
hist_neutral = (resultados[
    (resultados["home_score"].notna()) &
    (resultados["neutral"] == True) &
    (resultados["date"] >= WIN) &
    (resultados["date"] <  CUTOFF)        # corregido
].sort_values("date").reset_index(drop=True))

print(f"Partidos neutrales jugados 2010→2026-06-09: {len(hist_neutral):,}")

# --- Paso 2: pegar Elo (merge_asof, tope 18 meses, anti-fuga) -------
def pegar_elo(df, col_eq, col_sal):
    izq = df[["date", col_eq]].rename(columns={col_eq: "team"})
    out = pd.merge_asof(
        izq.sort_values("date"),
        elo_serie.rename(columns={"rating": col_sal}),
        on="date", by="team",
        direction="backward", tolerance=TOPE
    )
    return out[col_sal].values

hist_neutral["elo_h"] = pegar_elo(hist_neutral, "home_team", "elo_h")
hist_neutral["elo_a"] = pegar_elo(hist_neutral, "away_team", "elo_a")

cov      = hist_neutral["elo_h"].notna() & hist_neutral["elo_a"].notna()
m_neutral = hist_neutral[cov].copy()

print(f"Con Elo válido (tope 18 meses): {len(m_neutral):,} ({cov.mean()*100:.1f}%)")
print(f"Descartados                   : {(~cov).sum():,}")
assert cov.mean() > 0.50, f"Cobertura baja: {cov.mean()*100:.1f}%"

# --- Paso 3: formato largo ------------------------------------------
rows_h = pd.DataFrame({
    "goals":    m_neutral["home_score"].astype(int),
    "elo_diff": m_neutral["elo_h"] - m_neutral["elo_a"]
})
rows_a = pd.DataFrame({
    "goals":    m_neutral["away_score"].astype(int),
    "elo_diff": m_neutral["elo_a"] - m_neutral["elo_h"]
})
long_neutral = pd.concat([rows_h, rows_a], ignore_index=True)

print(f"\nlong_neutral: {len(long_neutral):,} filas ({len(m_neutral):,} × 2)")
print(f"  elo_diff — media={long_neutral['elo_diff'].mean():.1f} "
      f"| std={long_neutral['elo_diff'].std():.1f} "
      f"| min={long_neutral['elo_diff'].min():.0f} "
      f"| max={long_neutral['elo_diff'].max():.0f}")
print(f"  goals    — media={long_neutral['goals'].mean():.3f}")

# --- Paso 4: ajustar modelo Poisson neutral -------------------------
model_neutral = smf.glm(
    "goals ~ elo_diff",
    data=long_neutral,
    family=sm.families.Poisson()
).fit()

b_n = model_neutral.params["elo_diff"]
a_n = model_neutral.params["Intercept"]

assert b_n > 0,                  f"β negativo: {b_n:.6f}"
assert 0.9 < np.exp(a_n) < 2.0, f"exp(α) fuera de rango: {np.exp(a_n):.3f}"

print(f"\n--- Parámetros del modelo neutral ---")
print(f"  α = {a_n:.6f}  →  exp(α) = {np.exp(a_n):.3f} goles esperados (duelo parejo)")
print(f"  β = {b_n:.6f}  →  por cada 100 pts de ventaja Elo, "
      f"λ × {np.exp(b_n*100):.3f}")

# --- Paso 5: ejemplos de cordura ------------------------------------
r = elo_pre.set_index("country")["rating"]

def predecir(eq_h, eq_a):
    eh, ea = r[eq_h], r[eq_a]
    lh = np.exp(a_n + b_n*(eh-ea))
    la = np.exp(a_n + b_n*(ea-eh))
    M  = np.outer(poisson.pmf(np.arange(11), lh),
                  poisson.pmf(np.arange(11), la))
    return lh, la, np.tril(M,-1).sum(), np.trace(M), np.triu(M,1).sum()

print(f"\n--- Ejemplos de cordura ---")
print(f"  {'Partido':<28} {'λh':>5} {'λa':>5}  {'W':>5} {'D':>5} {'L':>5}")
for h, a in [("Spain","Qatar"),("France","Brazil"),
             ("Argentina","Australia"),("Mexico","Ecuador")]:
    lh, la, pw, pd_, pl = predecir(h, a)
    print(f"  {h+' vs '+a:<28} {lh:>5.2f} {la:>5.2f}  "
          f"{pw:>5.2f} {pd_:>5.2f} {pl:>5.2f}")

# --- Paso 6: guardar en Drive ---------------------------------------
cols_m = ["date","home_team","away_team","home_score","away_score",
          "tournament","neutral","elo_h","elo_a"]
m_neutral[cols_m].to_csv(f"{OUT}/m_neutral_2010_2026jun.csv", index=False)
long_neutral.to_csv(f"{OUT}/long_neutral.csv", index=False)

meta = {
    "creado": HOY,
    "version": "v2_neutral",
    "ventana_fit": "2010-01-01 → 2026-06-09",
    "filtro": "neutral==True",
    "tope_merge_asof_dias": 548,
    "formula": "goals ~ elo_diff (sin is_home)",
    "justificacion_neutral": (
        "El Mundial 2026 es contexto neutral. "
        "β estimado sobre neutrales no comparte señal con γ. "
        "Estable entre auxiliares 2018/2022/2026 (ratio 1.0-1.1x)."
    ),
    "alias_curaduria": alias,
    "n_partidos_neutrales": int(len(m_neutral)),
    "n_filas_long": int(len(long_neutral)),
    "cobertura_pct": round(float(cov.mean())*100, 2),
    "parametros": {
        "alpha": round(float(a_n), 6),
        "beta":  round(float(b_n), 6),
        "exp_alpha": round(float(np.exp(a_n)), 3)
    },
    "fuentes": {
        "resultados": "martj42/Kaggle (observed)",
        "elo_serie":  "saifalnimri/Kaggle, eloratings.net (borrowed)",
        "elo_pre":    "afonsofernandescruz/Kaggle, eloratings.net (borrowed)"
    }
}
with open(f"{OUT}/metadata_modelo_neutral.json","w",encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\nGuardado en {OUT}:")
for fn in ["m_neutral_2010_2026jun.csv","long_neutral.csv","metadata_modelo_neutral.json"]:
    kb = os.path.getsize(f"{OUT}/{fn}") / 1024
    print(f"  {fn:<40} {kb:7.1f} KB  {'OK' if kb>0 else 'VACÍO!'}")

print(f"\n✓ BLOQUE 2 completo")
print(f"  Variables en sesión: m_neutral, long_neutral, model_neutral")
print(f"  β = {b_n:.6f} | exp(α) = {np.exp(a_n):.3f}")

Partidos neutrales jugados 2010→2026-06-09: 4,814
Con Elo válido (tope 18 meses): 2,789 (57.9%)
Descartados                   : 2,025

long_neutral: 5,578 filas (2,789 × 2)
  elo_diff — media=0.0 | std=255.0 | min=-1013 | max=1013
  goals    — media=1.316

--- Parámetros del modelo neutral ---
  α = 0.247047  →  exp(α) = 1.280 goles esperados (duelo parejo)
  β = 0.000920  →  por cada 100 pts de ventaja Elo, λ × 1.096

--- Ejemplos de cordura ---
  Partido                         λh    λa      W     D     L
  Spain vs Qatar                2.53  0.65   0.78  0.14  0.07
  France vs Brazil              1.40  1.17   0.42  0.26  0.31
  Argentina vs Australia        1.73  0.94   0.56  0.24  0.20
  Mexico vs Ecuador             1.20  1.37   0.33  0.26  0.41

Guardado en /content/drive/MyDrive/wc2026_forecasting/v2:
  m_neutral_2010_2026jun.csv                 195.0 KB  OK
  long_neutral.csv                            44.5 KB  OK
  metadata_modelo_neutral.json                 0.8 KB  OK

✓ BLO

In [ ]:
# =====================================================================
# BLOQUE 3a — ESTRUCTURA DEL TORNEO 2026
# Inputs : equipos48 (Bloque 1)
# Outputs: TABLA_495, GRUPOS_2026, BRACKET_FIJOS, BRACKET_TERCEROS,
#          BRACKET_R16, BRACKET_QF, BRACKET_SF
# Guarda : tabla_495_combinaciones.csv en Drive
# Fuentes: Wikipedia (Anexo C FIFA), documento oficial FIFA
# =====================================================================
import requests, pandas as pd, json, os
from io import StringIO

for v in ["equipos48", "OUT"]:
    assert v in dir(), f"Falta '{v}': re-corre el Bloque 1"

# --- Grupos oficiales (fuente: sorteo FIFA 5-dic-2025) ---------------
GRUPOS_2026 = {
    "A": ["Czech Republic", "Mexico",               "South Africa",  "South Korea"],
    "B": ["Bosnia and Herzegovina", "Canada",        "Qatar",         "Switzerland"],
    "C": ["Brazil",   "Haiti",                       "Morocco",       "Scotland"],
    "D": ["Australia","Paraguay",                    "Turkey",        "United States"],
    "E": ["Curaçao",  "Ecuador",                     "Germany",       "Ivory Coast"],
    "F": ["Japan",    "Netherlands",                 "Sweden",        "Tunisia"],
    "G": ["Belgium",  "Egypt",                       "Iran",          "New Zealand"],
    "H": ["Cape Verde","Saudi Arabia",               "Spain",         "Uruguay"],
    "I": ["France",   "Iraq",                        "Norway",        "Senegal"],
    "J": ["Algeria",  "Argentina",                   "Austria",       "Jordan"],
    "K": ["Colombia", "DR Congo",                    "Portugal",      "Uzbekistan"],
    "L": ["Croatia",  "England",                     "Ghana",         "Panama"],
}

equipos_grupos = sorted(eq for grp in GRUPOS_2026.values() for eq in grp)
faltan_en_grupos = set(equipos48) - set(equipos_grupos)
faltan_en_48     = set(equipos_grupos) - set(equipos48)
assert not faltan_en_grupos, f"En equipos48 pero no en grupos: {faltan_en_grupos}"
assert not faltan_en_48,     f"En grupos pero no en equipos48: {faltan_en_48}"
print(f"✓ Grupos: 12 × 4 = {len(equipos_grupos)} equipos, todos verificados")

# --- Bracket (fuente: documento oficial FIFA/Wikipedia) --------------
BRACKET_FIJOS = {
    73: ("2A","2B"), 75: ("1F","2C"), 76: ("1C","2F"), 78: ("2E","2I"),
    83: ("2K","2L"), 84: ("1H","2J"), 86: ("1J","2H"), 88: ("2D","2G"),
}
BRACKET_TERCEROS = {
    74: ("1E","A/B/C/D/F"), 77: ("1I","C/D/F/G/H"),
    79: ("1A","C/E/F/H/I"), 80: ("1L","E/H/I/J/K"),
    81: ("1D","B/E/F/I/J"), 82: ("1G","A/E/H/I/J"),
    85: ("1B","E/F/G/I/J"), 87: ("1K","D/E/I/J/L"),
}
BRACKET_R16 = {
    89:(74,77), 90:(73,75), 91:(76,78), 92:(79,80),
    93:(83,84), 94:(81,82), 95:(86,88), 96:(85,87),
}
BRACKET_QF  = {97:(89,90), 98:(93,94), 99:(91,92), 100:(95,96)}
BRACKET_SF  = {101:(97,98), 102:(99,100)}
print(f"✓ Bracket: {len(BRACKET_FIJOS)} fijos + {len(BRACKET_TERCEROS)} con terceros")

# --- Tabla 495 combinaciones (Wikipedia, Anexo C FIFA) ---------------
URL     = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_knockout_stage"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
resp    = requests.get(URL, headers=headers)
assert resp.status_code == 200, f"Wikipedia devolvió {resp.status_code}"

tablas = pd.read_html(StringIO(resp.text))
raw    = tablas[0].copy()
assert raw.shape[0] == 495, f"Esperadas 495 filas, obtenidas {raw.shape[0]}"

GRUPO_COLS = [c for c in raw.columns if c.startswith("Third-placed")]
ASIGN_COLS = ["1A vs","1B vs","1D vs","1E vs","1G vs","1I vs","1K vs","1L vs"]
assert all(c in raw.columns for c in ASIGN_COLS), "Faltan columnas de asignación"

TABLA_495 = {}
registros = []
for _, row in raw.iterrows():
    grupos = frozenset(
        str(v).strip() for v in row[GRUPO_COLS].values
        if pd.notna(v) and str(v).strip() != "nan"
    )
    if len(grupos) != 8:
        continue
    asign = {col.replace(" vs",""): str(row[col]).strip()
             for col in ASIGN_COLS if str(row[col]).strip() != "nan"}
    TABLA_495[grupos] = asign
    registros.append({
        "grupos_clasificados": ",".join(sorted(grupos)),
        "1A": asign.get("1A",""), "1B": asign.get("1B",""),
        "1D": asign.get("1D",""), "1E": asign.get("1E",""),
        "1G": asign.get("1G",""), "1I": asign.get("1I",""),
        "1K": asign.get("1K",""), "1L": asign.get("1L",""),
    })

assert len(TABLA_495) == 495, f"Esperadas 495, obtenidas {len(TABLA_495)}"

# Test de sanidad
clave_1 = frozenset(["E","F","G","H","I","J","K","L"])
assert TABLA_495[clave_1]["1A"] == "3E", "Test combinación 1 falló"
print(f"✓ TABLA_495: {len(TABLA_495)} combinaciones parseadas y verificadas")

# --- Guardar en Drive ------------------------------------------------
df_495 = pd.DataFrame(registros)
df_495.to_csv(f"{OUT}/tabla_495_combinaciones.csv", index=False)
kb = os.path.getsize(f"{OUT}/tabla_495_combinaciones.csv") / 1024
print(f"\nGuardado: tabla_495_combinaciones.csv ({kb:.1f} KB)")
print("  Nota: guardada ahora porque Wikipedia puede editarse")
print("  una vez que inicien los partidos reales del torneo.")

print("\n✓ BLOQUE 3a completo")
print("  Variables en sesión: TABLA_495, GRUPOS_2026, BRACKET_FIJOS,")
print("  BRACKET_TERCEROS, BRACKET_R16, BRACKET_QF, BRACKET_SF")

✓ Grupos: 12 × 4 = 48 equipos, todos verificados
✓ Bracket: 8 fijos + 8 con terceros
✓ TABLA_495: 495 combinaciones parseadas y verificadas

Guardado: tabla_495_combinaciones.csv (20.3 KB)
  Nota: guardada ahora porque Wikipedia puede editarse
  una vez que inicien los partidos reales del torneo.

✓ BLOQUE 3a completo
  Variables en sesión: TABLA_495, GRUPOS_2026, BRACKET_FIJOS,
  BRACKET_TERCEROS, BRACKET_R16, BRACKET_QF, BRACKET_SF


In [ ]:
# =====================================================================
# BLOQUE 3b — MONTE CARLO 2026
# Inputs : a_n, b_n, elo_pre, TABLA_495, GRUPOS_2026, brackets
# Outputs: df_resultados (probabilidades por equipo)
# Guarda : mc2026_probabilidades.csv
# Supuesto S1: todo el torneo es neutral (sin ventaja de localía)
# Supuesto S2: empates en eliminatorias se resuelven 50/50 (penales)
# =====================================================================
import numpy as np, pandas as pd
from scipy.stats import poisson
import warnings
warnings.filterwarnings("ignore")

for v in ["a_n","b_n","elo_pre","TABLA_495","GRUPOS_2026",
          "BRACKET_FIJOS","BRACKET_TERCEROS",
          "BRACKET_R16","BRACKET_QF","BRACKET_SF","OUT"]:
    assert v in dir(), f"Falta '{v}': re-corre el bloque correspondiente"

N_SIM = 10_000
r     = elo_pre.set_index("country")["rating"]

# --- Funciones base --------------------------------------------------
def sim_partido(eq_h, eq_a):
    """Simula goles de un partido neutral."""
    eh = r.get(eq_h, 1700)
    ea = r.get(eq_a, 1700)
    lh = np.exp(a_n + b_n*(eh-ea))
    la = np.exp(a_n + b_n*(ea-eh))
    return np.random.poisson(lh), np.random.poisson(la)

def sim_eliminatorio(eq_h, eq_a):
    """Simula partido eliminatorio. Empate → 50/50 (penales)."""
    gh, ga = sim_partido(eq_h, eq_a)
    if gh != ga:
        return eq_h if gh > ga else eq_a
    return eq_h if np.random.random() < 0.5 else eq_a

# --- Fase de grupos --------------------------------------------------
def sim_grupos():
    primeros, segundos, terceros_info = {}, {}, {}
    for letra, equipos in GRUPOS_2026.items():
        tabla = {e: {"pts":0,"dg":0,"gf":0} for e in equipos}
        for i in range(len(equipos)):
            for j in range(i+1, len(equipos)):
                h, a = equipos[i], equipos[j]
                gh, ga = sim_partido(h, a)
                tabla[h]["gf"] += gh; tabla[a]["gf"] += ga
                tabla[h]["dg"] += gh-ga; tabla[a]["dg"] += ga-gh
                if gh > ga:    tabla[h]["pts"] += 3
                elif gh == ga: tabla[h]["pts"] += 1; tabla[a]["pts"] += 1
                else:          tabla[a]["pts"] += 3
        orden = sorted(equipos,
                       key=lambda e: (tabla[e]["pts"],
                                      tabla[e]["dg"],
                                      tabla[e]["gf"]),
                       reverse=True)
        primeros[letra] = orden[0]
        segundos[letra] = orden[1]
        terceros_info[letra] = {
            "equipo": orden[2],
            "pts":    tabla[orden[2]]["pts"],
            "dg":     tabla[orden[2]]["dg"],
            "gf":     tabla[orden[2]]["gf"],
        }
    return primeros, segundos, terceros_info

# --- 8 mejores terceros ---------------------------------------------
def seleccionar_terceros(terceros_info):
    df = pd.DataFrame(terceros_info).T.reset_index()
    df.columns = ["grupo","equipo","pts","dg","gf"]
    df = df.sort_values(["pts","dg","gf"], ascending=False).reset_index(drop=True)
    top8 = df.head(8)
    return frozenset(top8["grupo"].tolist()), dict(zip(top8["grupo"], top8["equipo"]))

# --- Resolver bracket Round of 32 -----------------------------------
def resolver_r32(primeros, segundos, grupos_cls, equipos_terceros):
    asign        = TABLA_495.get(grupos_cls, {})
    participantes = {}
    for num, (slot_h, slot_a) in BRACKET_FIJOS.items():
        eq_h = primeros[slot_h[1]] if slot_h[0]=="1" else segundos[slot_h[1]]
        eq_a = primeros[slot_a[1]] if slot_a[0]=="1" else segundos[slot_a[1]]
        participantes[num] = (eq_h, eq_a)
    for num, (slot_fijo, _) in BRACKET_TERCEROS.items():
        eq_fijo      = primeros[slot_fijo[1]]
        slot_tercero = asign.get(slot_fijo, None)
        if slot_tercero:
            grupo_tercero = slot_tercero[1]
            eq_tercero    = equipos_terceros.get(grupo_tercero,
                            list(equipos_terceros.values())[0])
        else:
            eq_tercero = list(equipos_terceros.values())[0]
        participantes[num] = (eq_fijo, eq_tercero)
    return participantes

# --- Simular bracket completo ----------------------------------------
def sim_bracket(participantes_r32):
    ganadores = {}
    for num, (h, a) in participantes_r32.items():
        ganadores[num] = sim_eliminatorio(h, a)
    for num, (m1, m2) in BRACKET_R16.items():
        ganadores[num] = sim_eliminatorio(ganadores[m1], ganadores[m2])
    for num, (m1, m2) in BRACKET_QF.items():
        ganadores[num] = sim_eliminatorio(ganadores[m1], ganadores[m2])
    for num, (m1, m2) in BRACKET_SF.items():
        ganadores[num] = sim_eliminatorio(ganadores[m1], ganadores[m2])
    campeon = sim_eliminatorio(ganadores[101], ganadores[102])
    return ganadores, campeon

# --- Loop principal --------------------------------------------------
conteo = {eq: {"r32":0,"r16":0,"qf":0,"sf":0,"campeon":0}
          for eq in equipos48}

for _ in range(N_SIM):
    primeros, segundos, terceros_info = sim_grupos()
    grupos_cls, equipos_terceros      = seleccionar_terceros(terceros_info)
    part_r32                          = resolver_r32(primeros, segundos,
                                                     grupos_cls, equipos_terceros)
    ganadores, campeon                = sim_bracket(part_r32)

    # Acumular por ronda
    for eq in (set(primeros.values()) | set(segundos.values()) |
               set(equipos_terceros.values())):
        conteo[eq]["r32"] += 1
    for m in [89,90,91,92,93,94,95,96]:
        conteo[ganadores[m]]["r16"] += 1
    for m in [97,98,99,100]:
        conteo[ganadores[m]]["qf"]  += 1
    for m in [101,102]:
        conteo[ganadores[m]]["sf"]  += 1
    conteo[campeon]["campeon"] += 1

# --- Tabla de resultados --------------------------------------------
filas = []
for eq in equipos48:
    c = conteo[eq]
    filas.append({
        "equipo":   eq,
        "elo":      int(r.get(eq, 0)),
        "p_r32":    round(c["r32"]    / N_SIM * 100, 1),
        "p_r16":    round(c["r16"]    / N_SIM * 100, 1),
        "p_qf":     round(c["qf"]     / N_SIM * 100, 1),
        "p_sf":     round(c["sf"]     / N_SIM * 100, 1),
        "p_campeon":round(c["campeon"]/ N_SIM * 100, 1),
    })

df_resultados = (pd.DataFrame(filas)
                   .sort_values("p_campeon", ascending=False)
                   .reset_index(drop=True))

# --- Reporte --------------------------------------------------------
print(f"{'='*62}")
print(f"  MONTE CARLO 2026 — {N_SIM:,} simulaciones")
print(f"  Modelo: neutral, β={b_n:.6f}, exp(α)={np.exp(a_n):.3f}")
print(f"  Supuestos: torneo neutral, penales 50/50")
print(f"{'='*62}")
print(f"\n{'Equipo':<25} {'Elo':>5} {'R32':>6} {'R16':>6} "
      f"{'QF':>6} {'SF':>6} {'🏆':>6}")
print("-"*59)
for _, row in df_resultados.iterrows():
    print(f"  {row['equipo']:<23} {row['elo']:>5} "
          f"{row['p_r32']:>5.1f}% {row['p_r16']:>5.1f}% "
          f"{row['p_qf']:>5.1f}% {row['p_sf']:>5.1f}% "
          f"{row['p_campeon']:>5.1f}%")

# Verificación: probabilidades de campeón suman ~100%
total_campeon = df_resultados["p_campeon"].sum()
print(f"\nSuma p_campeon: {total_campeon:.1f}% (esperado ~100%)")
assert 98 < total_campeon < 102, f"Probabilidades no suman ~100%: {total_campeon}"

# --- Guardar --------------------------------------------------------
df_resultados.to_csv(f"{OUT}/mc2026_probabilidades.csv", index=False)
kb = os.path.getsize(f"{OUT}/mc2026_probabilidades.csv") / 1024
print(f"Guardado: mc2026_probabilidades.csv ({kb:.1f} KB)")

print(f"\n✓ BLOQUE 3b completo")
print(f"  Variable en sesión: df_resultados")

  MONTE CARLO 2026 — 10,000 simulaciones
  Modelo: neutral, β=0.000920, exp(α)=1.280
  Supuestos: torneo neutral, penales 50/50

Equipo                      Elo    R32    R16     QF     SF      🏆
-----------------------------------------------------------
  Spain                    2165  95.3%  41.4%  27.9%  18.7%  12.1%
  Argentina                2113  90.6%  36.4%  23.2%  14.3%   8.5%
  France                   2081  88.0%  35.2%  21.9%  12.7%   7.8%
  England                  2020  89.2%  31.8%  18.9%  10.4%   5.7%
  Portugal                 1984  82.9%  27.2%  15.8%   8.5%   4.7%
  Brazil                   1984  87.7%  29.6%  16.6%   8.8%   4.6%
  Colombia                 1975  81.9%  26.2%  14.5%   8.0%   4.1%
  Netherlands              1961  83.6%  27.2%  14.4%   7.3%   3.8%
  Germany                  1923  86.1%  24.5%  12.9%   6.2%   3.3%
  Ecuador                  1933  86.7%  24.4%  13.1%   6.7%   3.3%
  Switzerland              1889  88.4%  26.2%  12.5%   6.1%   3.1%
  Turke

In [ ]:
# =====================================================================
# BOOTSTRAP — reanuda sesión desde Drive
# Corre esta celda SIEMPRE al inicio de una sesión nueva.
# Lee desde Drive lo que puede; te dice qué falta si algo no existe.
# No re-descarga Kaggle ni recalcula nada.
# =====================================================================
import os, json, pandas as pd, numpy as np
from scipy.stats import poisson
from io import StringIO

# --- 1) Montar Drive -------------------------------------------------
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive ya montado.")

OUT  = "/content/drive/MyDrive/wc2026_forecasting/v2"
falta_algo = False

def check(archivo):
    ruta = f"{OUT}/{archivo}"
    ok   = os.path.exists(ruta) and os.path.getsize(ruta) > 0
    estado = "✓" if ok else "✗ FALTA"
    print(f"  {estado}  {archivo}")
    return ok

print("\n--- Archivos en Drive ---")
archivos_ok = {
    "elo_pre":      check("elo_pre_48.csv"),
    "m_neutral":    check("m_neutral_2010_2026jun.csv"),
    "long_neutral": check("long_neutral.csv"),
    "metadata":     check("metadata_modelo_neutral.json"),
    "tabla_495":    check("tabla_495_combinaciones.csv"),
    "simulacion":   check("mc2026_probabilidades.csv"),
}

# --- 2) Recargar datos desde Drive -----------------------------------
print("\n--- Recargando variables ---")

if archivos_ok["elo_pre"]:
    elo_pre   = pd.read_csv(f"{OUT}/elo_pre_48.csv",
                             parse_dates=["snapshot_date"])
    equipos48 = sorted(elo_pre["country"].tolist())
    alias     = {"Czechia": "Czech Republic"}
    print(f"  ✓ elo_pre       ({len(elo_pre)} equipos)")
    print(f"  ✓ equipos48     ({len(equipos48)} equipos)")
    print(f"  ✓ alias         {alias}")
else:
    print("  ✗ elo_pre no disponible → re-corre Bloque 1")
    falta_algo = True

if archivos_ok["m_neutral"]:
    m_neutral = pd.read_csv(f"{OUT}/m_neutral_2010_2026jun.csv",
                             parse_dates=["date"])
    print(f"  ✓ m_neutral     ({len(m_neutral):,} partidos)")
else:
    print("  ✗ m_neutral no disponible → re-corre Bloque 2")
    falta_algo = True

if archivos_ok["long_neutral"]:
    long_neutral = pd.read_csv(f"{OUT}/long_neutral.csv")
    print(f"  ✓ long_neutral  ({len(long_neutral):,} filas)")
else:
    print("  ✗ long_neutral no disponible → re-corre Bloque 2")
    falta_algo = True

if archivos_ok["metadata"]:
    with open(f"{OUT}/metadata_modelo_neutral.json", encoding="utf-8") as f:
        meta = json.load(f)
    a_n = meta["parametros"]["alpha"]
    b_n = meta["parametros"]["beta"]
    print(f"  ✓ model_neutral β={b_n:.6f} | exp(α)={np.exp(a_n):.3f}")
else:
    print("  ✗ metadata no disponible → re-corre Bloque 2")
    falta_algo = True

if archivos_ok["simulacion"]:
    df_resultados = pd.read_csv(f"{OUT}/mc2026_probabilidades.csv")
    print(f"  ✓ df_resultados ({len(df_resultados)} equipos)")
else:
    print("  ✗ df_resultados no disponible → re-corre Bloque 3b")
    falta_algo = True

# --- 3) Función predecir (depende de a_n, b_n, elo_pre) -------------
if archivos_ok["elo_pre"] and archivos_ok["metadata"]:
    r = elo_pre.set_index("country")["rating"]
    def predecir(eq_h, eq_a):
        eh, ea = r.get(eq_h, 1700), r.get(eq_a, 1700)
        lh = np.exp(a_n + b_n*(eh-ea))
        la = np.exp(a_n + b_n*(ea-eh))
        M  = np.outer(poisson.pmf(np.arange(11), lh),
                      poisson.pmf(np.arange(11), la))
        return lh, la, np.tril(M,-1).sum(), np.trace(M), np.triu(M,1).sum()
    print(f"  ✓ predecir()    lista")

# --- 4) TABLA_495 desde CSV (no necesita Wikipedia) -----------------
if archivos_ok["tabla_495"]:
    df_495   = pd.read_csv(f"{OUT}/tabla_495_combinaciones.csv")
    ASIGN_COLS = ["1A","1B","1D","1E","1G","1I","1K","1L"]
    TABLA_495  = {}
    for _, row in df_495.iterrows():
        grupos = frozenset(row["grupos_clasificados"].split(","))
        asign  = {col: str(row[col]).strip()
                  for col in ASIGN_COLS
                  if pd.notna(row[col]) and str(row[col]).strip() != "nan"}
        TABLA_495[grupos] = asign
    assert len(TABLA_495) == 495
    print(f"  ✓ TABLA_495     ({len(TABLA_495)} combinaciones)")
else:
    print("  ✗ TABLA_495 no disponible → re-corre Bloque 3a")
    falta_algo = True

# --- 5) Estructura del torneo (código puro, instantáneo) ------------
GRUPOS_2026 = {
    "A": ["Czech Republic","Mexico",              "South Africa","South Korea"],
    "B": ["Bosnia and Herzegovina","Canada",       "Qatar",       "Switzerland"],
    "C": ["Brazil",  "Haiti",                      "Morocco",     "Scotland"],
    "D": ["Australia","Paraguay",                  "Turkey",      "United States"],
    "E": ["Curaçao", "Ecuador",                    "Germany",     "Ivory Coast"],
    "F": ["Japan",   "Netherlands",                "Sweden",      "Tunisia"],
    "G": ["Belgium", "Egypt",                      "Iran",        "New Zealand"],
    "H": ["Cape Verde","Saudi Arabia",             "Spain",       "Uruguay"],
    "I": ["France",  "Iraq",                       "Norway",      "Senegal"],
    "J": ["Algeria", "Argentina",                  "Austria",     "Jordan"],
    "K": ["Colombia","DR Congo",                   "Portugal",    "Uzbekistan"],
    "L": ["Croatia", "England",                    "Ghana",       "Panama"],
}
BRACKET_FIJOS = {
    73:("2A","2B"), 75:("1F","2C"), 76:("1C","2F"), 78:("2E","2I"),
    83:("2K","2L"), 84:("1H","2J"), 86:("1J","2H"), 88:("2D","2G"),
}
BRACKET_TERCEROS = {
    74:("1E","A/B/C/D/F"), 77:("1I","C/D/F/G/H"),
    79:("1A","C/E/F/H/I"), 80:("1L","E/H/I/J/K"),
    81:("1D","B/E/F/I/J"), 82:("1G","A/E/H/I/J"),
    85:("1B","E/F/G/I/J"), 87:("1K","D/E/I/J/L"),
}
BRACKET_R16 = {
    89:(74,77), 90:(73,75), 91:(76,78), 92:(79,80),
    93:(83,84), 94:(81,82), 95:(86,88), 96:(85,87),
}
BRACKET_QF  = {97:(89,90), 98:(93,94), 99:(91,92), 100:(95,96)}
BRACKET_SF  = {101:(97,98), 102:(99,100)}
print(f"  ✓ GRUPOS_2026 + brackets hardcodeados")

# --- 6) Nota sobre variables que requieren Kaggle -------------------
print(f"\n--- Variables que requieren Kaggle (no en Drive) ---")
print(f"  resultados, elo_serie → re-corre Bloque 1 si las necesitas")
print(f"  (solo necesarias para reajustar el modelo con nuevos datos)")

# --- 7) Resumen final ------------------------------------------------
print(f"\n{'='*45}")
if not falta_algo:
    print(f"  ✓ Sesión completamente reanudada")
    print(f"  Puedes usar predecir(), df_resultados,")
    print(f"  TABLA_495 y brackets directamente.")
else:
    print(f"  ✗ Faltan archivos — revisa los mensajes arriba")
print(f"{'='*45}")

Drive ya montado.

--- Archivos en Drive ---
  ✓  elo_pre_48.csv
  ✓  m_neutral_2010_2026jun.csv
  ✓  long_neutral.csv
  ✓  metadata_modelo_neutral.json
  ✓  tabla_495_combinaciones.csv
  ✓  mc2026_probabilidades.csv

--- Recargando variables ---
  ✓ elo_pre       (48 equipos)
  ✓ equipos48     (48 equipos)
  ✓ alias         {'Czechia': 'Czech Republic'}
  ✓ m_neutral     (2,789 partidos)
  ✓ long_neutral  (5,578 filas)
  ✓ model_neutral β=0.000920 | exp(α)=1.280
  ✓ df_resultados (48 equipos)
  ✓ predecir()    lista
  ✓ TABLA_495     (495 combinaciones)
  ✓ GRUPOS_2026 + brackets hardcodeados

--- Variables que requieren Kaggle (no en Drive) ---
  resultados, elo_serie → re-corre Bloque 1 si las necesitas
  (solo necesarias para reajustar el modelo con nuevos datos)

  ✓ Sesión completamente reanudada
  Puedes usar predecir(), df_resultados,
  TABLA_495 y brackets directamente.


In [ ]:
# Mover elo_pre_48.csv a la carpeta correcta
import shutil, os

origen  = "/content/drive/MyDrive/wc2026_forecasting/v2/elo_pre_48.csv"
destino = f"{OUT}/elo_pre_48.csv"

# Buscar dónde está realmente
posibles = [
    "/content/drive/MyDrive/wc2026_forecasting/elo_pre_48.csv",
    "/content/drive/MyDrive/wc2026_forecasting/v2/datos/elo_pre_48.csv",
    "/content/drive/MyDrive/wc2026_forecasting/v2/elo_pre_48.csv",
]
for p in posibles:
    if os.path.exists(p):
        print(f"Encontrado en: {p}")
        if p != destino:
            shutil.copy(p, destino)
            print(f"Copiado a: {destino}")
        else:
            print("Ya está en la ruta correcta")
        break
else:
    print("No encontrado en rutas conocidas")
    print("Contenido de v2/:")
    print(os.listdir(OUT))

Encontrado en: /content/drive/MyDrive/wc2026_forecasting/v2/datos/elo_pre_48.csv
Copiado a: /content/drive/MyDrive/wc2026_forecasting/v2/elo_pre_48.csv


In [ ]:
# Verificación rápida post-copia
import os
OUT = "/content/drive/MyDrive/wc2026_forecasting/v2"
archivos = ["elo_pre_48.csv","m_neutral_2010_2026jun.csv",
            "long_neutral.csv","metadata_modelo_neutral.json",
            "tabla_495_combinaciones.csv","mc2026_probabilidades.csv"]
print("--- Verificación archivos en Drive ---")
for fn in archivos:
    ruta = f"{OUT}/{fn}"
    ok   = os.path.exists(ruta) and os.path.getsize(ruta) > 0
    kb   = os.path.getsize(ruta)/1024 if ok else 0
    print(f"  {'✓' if ok else '✗'}  {fn:<40} {kb:6.1f} KB")

--- Verificación archivos en Drive ---
  ✓  elo_pre_48.csv                              1.6 KB
  ✓  m_neutral_2010_2026jun.csv                195.0 KB
  ✓  long_neutral.csv                           44.5 KB
  ✓  metadata_modelo_neutral.json                0.8 KB
  ✓  tabla_495_combinaciones.csv                20.3 KB
  ✓  mc2026_probabilidades.csv                   1.7 KB


In [ ]:
# =====================================================================
# BLOQUE 3 — VALIDACIÓN
# Inputs : resultados, elo_serie, OUT
# Outputs: df_validacion, métricas por torneo
# Guarda : validacion_2018_2022.csv
# Modelo : neutral, sin localía — igual que Bloque 2
# Cortes : 2018-06-14 y 2022-11-20 (por fecha, no por etiqueta)
# =====================================================================
import pandas as pd, numpy as np, warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import poisson
warnings.filterwarnings("ignore")

for v in ["resultados","elo_serie","OUT"]:
    assert v in dir(), f"Falta '{v}': re-corre Bloque 1"

TOPE = pd.Timedelta(days=548)   # 18 meses

# --- Funciones base --------------------------------------------------
def pegar_elo(df, col_eq, col_sal, tope=TOPE):
    izq = df[["date", col_eq]].rename(columns={col_eq:"team"})
    out = pd.merge_asof(
        izq.sort_values("date"),
        elo_serie.rename(columns={"rating":col_sal}),
        on="date", by="team",
        direction="backward", tolerance=tope
    )
    return out[col_sal].values

def ajustar_modelo_neutral(corte):
    """Ajusta Poisson sobre partidos neutrales antes del corte."""
    hist = resultados[
        (resultados["home_score"].notna()) &
        (resultados["neutral"] == True) &
        (resultados["date"] >= pd.Timestamp("2010-01-01")) &
        (resultados["date"] <  corte)
    ].sort_values("date").reset_index(drop=True)

    hist["elo_h"] = pegar_elo(hist, "home_team", "elo_h")
    hist["elo_a"] = pegar_elo(hist, "away_team", "elo_a")
    hist = hist[hist["elo_h"].notna() & hist["elo_a"].notna()].copy()

    h = pd.DataFrame({"goals": hist["home_score"].astype(int),
                       "elo_diff": hist["elo_h"] - hist["elo_a"]})
    a = pd.DataFrame({"goals": hist["away_score"].astype(int),
                       "elo_diff": hist["elo_a"] - hist["elo_h"]})
    lng = pd.concat([h, a], ignore_index=True)

    mdl = smf.glm("goals ~ elo_diff", data=lng,
                  family=sm.families.Poisson()).fit()
    return mdl, len(hist)

def predecir_partido(elo_h, elo_a, params, maxg=10):
    """Devuelve (lambda_h, lambda_a, P_win, P_draw, P_loss)."""
    a_, b_ = params["Intercept"], params["elo_diff"]
    lh = np.exp(a_ + b_*(elo_h-elo_a))
    la = np.exp(a_ + b_*(elo_a-elo_h))
    rng = np.arange(maxg+1)
    M   = np.outer(poisson.pmf(rng, lh), poisson.pmf(rng, la))
    return lh, la, np.tril(M,-1).sum(), np.trace(M), np.triu(M,1).sum()

def get_elo_torneo(equipo, corte, tope=TOPE):
    """Elo más reciente antes del corte. Sin tope si no hay dentro."""
    s = elo_serie[
        (elo_serie["team"]==equipo) &
        (elo_serie["date"] >= corte-tope) &
        (elo_serie["date"] <  corte)
    ]
    if len(s)==0:
        s = elo_serie[
            (elo_serie["team"]==equipo) &
            (elo_serie["date"] <  corte)
        ]
    if len(s)==0:
        return np.nan, None
    row = s.sort_values("date").iloc[-1]
    return float(row["rating"]), row["date"]

def brier(probs, res):
    return sum((p-r)**2 for p,r in zip(probs,res))

def logloss(probs, res):
    eps = 1e-10
    return -sum(r*np.log(p+eps) for p,r in zip(probs,res))

def rps(probs, res):
    cp = np.cumsum(probs); cr = np.cumsum(res)
    return sum((a-b)**2 for a,b in zip(cp[:-1],cr[:-1]))/(len(probs)-1)

def res_vector(gh, ga):
    if gh>ga:   return [1,0,0]
    elif gh==ga: return [0,1,0]
    else:        return [0,0,1]

# --- Loop por torneo -------------------------------------------------
TORNEOS = {
    2018: pd.Timestamp("2018-06-14"),
    2022: pd.Timestamp("2022-11-20"),
}
REF = [1/3, 1/3, 1/3]

todos_registros = []

for anio, corte in TORNEOS.items():
    print(f"\n{'='*55}")
    print(f"  VALIDACIÓN {anio} — corte: {corte.date()}")
    print(f"{'='*55}")

    # Modelo auxiliar neutral
    mdl, n_fit = ajustar_modelo_neutral(corte)
    params = mdl.params
    print(f"  Partidos neutrales en fit: {n_fit:,}")
    print(f"  β={params['elo_diff']:.6f} | exp(α)={np.exp(params['Intercept']):.3f}")

    # Partidos del torneo con marcador real
    torneo = resultados[
        (resultados["tournament"]=="FIFA World Cup") &
        (resultados["date"].dt.year==anio) &
        (resultados["home_score"].notna())
    ].copy()

    # Pegar Elo pre-torneo a cada equipo
    equipos_t = set(torneo["home_team"]) | set(torneo["away_team"])
    elo_map   = {}
    advertencias = []
    for eq in equipos_t:
        rating, fecha = get_elo_torneo(eq, corte)
        elo_map[eq] = rating
        if fecha is not None:
            meses = (corte - fecha).days / 30
            if meses > 18:
                advertencias.append(f"    {eq}: Elo={rating:.0f} "
                                    f"(snapshot {fecha.date()}, "
                                    f"{meses:.0f} meses antes del corte)")

    if advertencias:
        print(f"\n  Equipos con Elo fuera del tope estándar (>18 meses):")
        for a in advertencias:
            print(a)

    torneo["elo_h"] = torneo["home_team"].map(elo_map)
    torneo["elo_a"] = torneo["away_team"].map(elo_map)
    torneo = torneo[torneo["elo_h"].notna() & torneo["elo_a"].notna()].copy()
    print(f"\n  Partidos evaluados: {len(torneo)} / 64")

    # Métricas partido a partido
    bs_l, ll_l, rps_l = [], [], []
    ref_bs_l, ref_ll_l, ref_rps_l = [], [], []

    for _, row in torneo.iterrows():
        lh, la, pw, pd_, pl = predecir_partido(
            row["elo_h"], row["elo_a"], params)
        probs = [pw, pd_, pl]
        res   = res_vector(int(row["home_score"]), int(row["away_score"]))

        bs_l.append(brier(probs, res))
        ll_l.append(logloss(probs, res))
        rps_l.append(rps(probs, res))

        ref_bs_l.append(brier(REF, res))
        ref_ll_l.append(logloss(REF, res))
        ref_rps_l.append(rps(REF, res))

        todos_registros.append({
            "torneo":      anio,
            "fecha":       row["date"].date(),
            "home_team":   row["home_team"],
            "away_team":   row["away_team"],
            "home_score":  int(row["home_score"]),
            "away_score":  int(row["away_score"]),
            "elo_h":       round(row["elo_h"],0),
            "elo_a":       round(row["elo_a"],0),
            "lambda_h":    round(lh,3),
            "lambda_a":    round(la,3),
            "p_home":      round(pw,4),
            "p_draw":      round(pd_,4),
            "p_away":      round(pl,4),
            "resultado":   "H" if res==[1,0,0] else ("D" if res==[0,1,0] else "A"),
            "brier":       round(brier(probs,res),4),
            "logloss":     round(logloss(probs,res),4),
            "rps":         round(rps(probs,res),4),
        })

    print(f"\n  {'Métrica':<12} {'Modelo':>10} {'Referencia':>12} {'Mejor?':>8}")
    print(f"  {'-'*44}")
    for nombre, vals, refs in [
        ("Brier",    bs_l,  ref_bs_l),
        ("Log Loss", ll_l,  ref_ll_l),
        ("RPS",      rps_l, ref_rps_l),
    ]:
        m_val = np.mean(vals)
        r_val = np.mean(refs)
        mejor = "✓" if m_val < r_val else "✗"
        print(f"  {nombre:<12} {m_val:>10.4f} {r_val:>12.4f} {mejor:>8}")

# --- Resumen conjunto ------------------------------------------------
df_validacion = pd.DataFrame(todos_registros)

print(f"\n{'='*55}")
print(f"  RESUMEN CONJUNTO")
print(f"{'='*55}")
print(f"  {'Torneo':<8} {'Brier':>8} {'LogLoss':>10} {'RPS':>8} {'N':>6}")
for anio in [2018, 2022]:
    sub = df_validacion[df_validacion["torneo"]==anio]
    print(f"  {anio:<8} {sub['brier'].mean():>8.4f} "
          f"{sub['logloss'].mean():>10.4f} "
          f"{sub['rps'].mean():>8.4f} "
          f"{len(sub):>6}")
print(f"\n  Referencia ingenua: Brier≈0.6667 | LogLoss≈1.099 | RPS≈0.2778")

# --- Guardar ---------------------------------------------------------
df_validacion.to_csv(f"{OUT}/validacion_2018_2022.csv", index=False)
kb = os.path.getsize(f"{OUT}/validacion_2018_2022.csv") / 1024
print(f"\nGuardado: validacion_2018_2022.csv ({kb:.1f} KB)")

print(f"\n✓ BLOQUE 3 completo")
print(f"  Variable en sesión: df_validacion ({len(df_validacion)} partidos)")


  VALIDACIÓN 2018 — corte: 2018-06-14
  Partidos neutrales en fit: 1,170
  β=0.001206 | exp(α)=1.292

  Partidos evaluados: 64 / 64

  Métrica          Modelo   Referencia   Mejor?
  --------------------------------------------
  Brier            0.5802       0.6667        ✓
  Log Loss         0.9763       1.0986        ✓
  RPS              0.2080       0.2439        ✓

  VALIDACIÓN 2022 — corte: 2022-11-20
  Partidos neutrales en fit: 1,834
  β=0.001146 | exp(α)=1.251

  Equipos con Elo fuera del tope estándar (>18 meses):
    Spain: Elo=2034 (snapshot 2021-03-28, 20 meses antes del corte)

  Partidos evaluados: 64 / 64

  Métrica          Modelo   Referencia   Mejor?
  --------------------------------------------
  Brier            0.5988       0.6667        ✓
  Log Loss         1.0150       1.0986        ✓
  RPS              0.2123       0.2387        ✓

  RESUMEN CONJUNTO
  Torneo      Brier    LogLoss      RPS      N
  2018       0.5802     0.9763   0.2080     64
  2022       0.5

In [ ]:
# =====================================================================
# BLOQUE — MONTE CARLO 2018 y 2022
# Inputs : resultados, elo_serie, OUT
# Outputs: df_mc2018, df_mc2022
# Guarda : mc2018_probabilidades.csv, mc2022_probabilidades.csv
# Modelo : auxiliar neutral con corte temporal limpio
# Bracket: hardcodeado (32 equipos, octavos directo)
# Penales: 50/50
# N      : 10,000 simulaciones por torneo
# =====================================================================
import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import poisson
warnings.filterwarnings("ignore")

for v in ["resultados","elo_serie","OUT"]:
    assert v in dir(), f"Falta '{v}': re-corre Bloque 1"

N_SIM = 10_000
TOPE  = pd.Timedelta(days=548)

# --- Estructura de torneos (hardcodeada) -----------------------------
GRUPOS = {
    2018: {
        "A": ["Russia","Saudi Arabia","Egypt","Uruguay"],
        "B": ["Portugal","Spain","Morocco","Iran"],
        "C": ["France","Australia","Peru","Denmark"],
        "D": ["Argentina","Iceland","Croatia","Nigeria"],
        "E": ["Brazil","Switzerland","Costa Rica","Serbia"],
        "F": ["Germany","Mexico","Sweden","South Korea"],
        "G": ["Belgium","Panama","Tunisia","England"],
        "H": ["Poland","Senegal","Colombia","Japan"],
    },
    2022: {
        "A": ["Qatar","Ecuador","Senegal","Netherlands"],
        "B": ["England","Iran","United States","Wales"],
        "C": ["Argentina","Saudi Arabia","Mexico","Poland"],
        "D": ["France","Australia","Denmark","Tunisia"],
        "E": ["Spain","Costa Rica","Germany","Japan"],
        "F": ["Belgium","Canada","Morocco","Croatia"],
        "G": ["Brazil","Serbia","Switzerland","Cameroon"],
        "H": ["Portugal","Ghana","Uruguay","South Korea"],
    }
}

# Bracket octavos: (1A vs 2B), (1C vs 2D), etc.
# Formato: lista de pares de grupos donde 1ro del primero vs 2do del segundo
BRACKET_OCTAVOS = {
    2018: [("A","B"),("C","D"),("E","F"),("G","H")],
    2022: [("A","B"),("C","D"),("E","F"),("G","H")],
}

# Resultados reales para comparación
REALES = {
    2018: {
        "campeon":       "France",
        "finalista":     "Croatia",
        "semifinalistas":{"France","Croatia","Belgium","England"},
        "cuartos":       {"France","Croatia","Belgium","England",
                          "Brazil","Uruguay","Russia","Sweden"},
        "octavos":       {"France","Argentina","Uruguay","Portugal",
                          "Spain","Russia","Croatia","Denmark",
                          "Brazil","Mexico","Belgium","Japan",
                          "Sweden","Switzerland","Colombia","England"},
    },
    2022: {
        "campeon":       "Argentina",
        "finalista":     "France",
        "semifinalistas":{"Argentina","France","Croatia","Morocco"},
        "cuartos":       {"Argentina","France","Croatia","Morocco",
                          "Netherlands","Brazil","England","Portugal"},
        "octavos":       {"Netherlands","Senegal","England","United States",
                          "Argentina","Poland","France","Australia",
                          "Morocco","Croatia","Brazil","South Korea",
                          "Portugal","Switzerland","Spain","Japan"},
    }
}

# --- Funciones base --------------------------------------------------
def pegar_elo_equipo(equipo, corte, tope=TOPE):
    s = elo_serie[
        (elo_serie["team"]==equipo) &
        (elo_serie["date"] >= corte-tope) &
        (elo_serie["date"] <  corte)
    ]
    if len(s)==0:
        s = elo_serie[
            (elo_serie["team"]==equipo) &
            (elo_serie["date"] <  corte)
        ]
    return float(s.sort_values("date").iloc[-1]["rating"]) if len(s)>0 else 1500.0

def ajustar_modelo_neutral(corte):
    hist = resultados[
        (resultados["home_score"].notna()) &
        (resultados["neutral"]==True) &
        (resultados["date"] >= pd.Timestamp("2010-01-01")) &
        (resultados["date"] <  corte)
    ].sort_values("date").reset_index(drop=True)

    hist["elo_h"] = pd.merge_asof(
        hist[["date","home_team"]].rename(columns={"home_team":"team"}),
        elo_serie.rename(columns={"rating":"elo_h"}),
        on="date", by="team", direction="backward", tolerance=TOPE
    )["elo_h"].values
    hist["elo_a"] = pd.merge_asof(
        hist[["date","away_team"]].rename(columns={"away_team":"team"}),
        elo_serie.rename(columns={"rating":"elo_a"}),
        on="date", by="team", direction="backward", tolerance=TOPE
    )["elo_a"].values
    hist = hist[hist["elo_h"].notna() & hist["elo_a"].notna()].copy()

    h   = pd.DataFrame({"goals": hist["home_score"].astype(int),
                         "elo_diff": hist["elo_h"]-hist["elo_a"]})
    a   = pd.DataFrame({"goals": hist["away_score"].astype(int),
                         "elo_diff": hist["elo_a"]-hist["elo_h"]})
    lng = pd.concat([h,a], ignore_index=True)
    mdl = smf.glm("goals ~ elo_diff", data=lng,
                  family=sm.families.Poisson()).fit()
    return mdl, len(hist)

def sim_partido_n(eq_h, eq_a, elos, a_, b_):
    eh = elos.get(eq_h, 1500); ea = elos.get(eq_a, 1500)
    lh = np.exp(a_ + b_*(eh-ea))
    la = np.exp(a_ + b_*(ea-eh))
    return np.random.poisson(lh), np.random.poisson(la)

def sim_eliminatorio_n(eq_h, eq_a, elos, a_, b_):
    gh, ga = sim_partido_n(eq_h, eq_a, elos, a_, b_)
    if gh!=ga: return eq_h if gh>ga else eq_a
    return eq_h if np.random.random()<0.5 else eq_a

def sim_grupos_n(grupos, elos, a_, b_):
    primeros, segundos = {}, {}
    for letra, equipos in grupos.items():
        tabla = {e:{"pts":0,"dg":0,"gf":0} for e in equipos}
        for i in range(len(equipos)):
            for j in range(i+1,len(equipos)):
                h,a = equipos[i],equipos[j]
                gh,ga = sim_partido_n(h,a,elos,a_,b_)
                tabla[h]["gf"]+=gh; tabla[a]["gf"]+=ga
                tabla[h]["dg"]+=gh-ga; tabla[a]["dg"]+=ga-gh
                if gh>ga:    tabla[h]["pts"]+=3
                elif gh==ga: tabla[h]["pts"]+=1; tabla[a]["pts"]+=1
                else:        tabla[a]["pts"]+=3
        orden = sorted(equipos,
                       key=lambda e:(tabla[e]["pts"],
                                     tabla[e]["dg"],
                                     tabla[e]["gf"]),
                       reverse=True)
        primeros[letra]=orden[0]; segundos[letra]=orden[1]
    return primeros, segundos

def sim_torneo_n(grupos, bracket_oct, elos, a_, b_):
    primeros, segundos = sim_grupos_n(grupos, elos, a_, b_)
    clasificados = set(primeros.values()) | set(segundos.values())

    # Octavos
    cuartos = []
    for g1,g2 in bracket_oct:
        cuartos.append(sim_eliminatorio_n(primeros[g1],segundos[g2],elos,a_,b_))
        cuartos.append(sim_eliminatorio_n(primeros[g2],segundos[g1],elos,a_,b_))

    # Cuartos
    semis = []
    for i in range(0,len(cuartos),2):
        semis.append(sim_eliminatorio_n(cuartos[i],cuartos[i+1],elos,a_,b_))

    # Semis
    finalistas = []
    for i in range(0,len(semis),2):
        finalistas.append(sim_eliminatorio_n(semis[i],semis[i+1],elos,a_,b_))

    # Final
    campeon = sim_eliminatorio_n(finalistas[0],finalistas[1],elos,a_,b_)

    return {
        "octavos":       clasificados,
        "cuartos":       set(cuartos),
        "semis":         set(semis),
        "finalistas":    set(finalistas),
        "campeon":       campeon,
    }

# --- Loop principal --------------------------------------------------
CORTES = {2018: pd.Timestamp("2018-06-14"),
          2022: pd.Timestamp("2022-11-20")}

resultados_mc = {}

for anio, corte in CORTES.items():
    print(f"\n{'='*58}")
    print(f"  MONTE CARLO {anio} — {N_SIM:,} simulaciones")
    print(f"{'='*58}")

    # Modelo auxiliar neutral
    mdl, n_fit = ajustar_modelo_neutral(corte)
    a_ = mdl.params["Intercept"]
    b_ = mdl.params["elo_diff"]
    print(f"  Partidos neutrales en fit: {n_fit:,}")
    print(f"  β={b_:.6f} | exp(α)={np.exp(a_):.3f}")

    # Elos pre-torneo
    grupos  = GRUPOS[anio]
    equipos = [eq for grp in grupos.values() for eq in grp]
    elos    = {eq: pegar_elo_equipo(eq, corte) for eq in equipos}

    # Advertencias Elo fuera de tope
    for eq in equipos:
        s = elo_serie[
            (elo_serie["team"]==eq) &
            (elo_serie["date"] >= corte-TOPE) &
            (elo_serie["date"] <  corte)
        ]
        if len(s)==0:
            s2 = elo_serie[
                (elo_serie["team"]==eq) &
                (elo_serie["date"] <  corte)
            ]
            if len(s2)>0:
                fecha = s2.sort_values("date").iloc[-1]["date"]
                meses = (corte-fecha).days/30
                print(f"  ⚠ {eq}: snapshot {fecha.date()} ({meses:.0f} meses)")

    # Simulaciones
    conteo = {eq:{"octavos":0,"cuartos":0,"semis":0,
                  "finalista":0,"campeon":0} for eq in equipos}

    for _ in range(N_SIM):
        res = sim_torneo_n(grupos, BRACKET_OCTAVOS[anio], elos, a_, b_)
        for eq in res["octavos"]:   conteo[eq]["octavos"]  +=1
        for eq in res["cuartos"]:   conteo[eq]["cuartos"]  +=1
        for eq in res["semis"]:     conteo[eq]["semis"]    +=1
        for eq in res["finalistas"]:conteo[eq]["finalista"]+=1
        conteo[res["campeon"]]["campeon"]+=1

    # Tabla de probabilidades
    reales = REALES[anio]
    filas  = []
    for eq in equipos:
        c = conteo[eq]
        filas.append({
            "equipo":      eq,
            "elo":         int(elos[eq]),
            "p_octavos":   round(c["octavos"] /N_SIM*100,1),
            "p_cuartos":   round(c["cuartos"] /N_SIM*100,1),
            "p_semis":     round(c["semis"]   /N_SIM*100,1),
            "p_finalista": round(c["finalista"]/N_SIM*100,1),
            "p_campeon":   round(c["campeon"] /N_SIM*100,1),
            "llego_octavos":   eq in reales["octavos"],
            "llego_cuartos":   eq in reales["cuartos"],
            "llego_semis":     eq in reales["semifinalistas"],
            "llego_final":     eq in {reales["campeon"],reales["finalista"]},
            "campeon_real":    eq == reales["campeon"],
        })

    df = (pd.DataFrame(filas)
            .sort_values("p_campeon", ascending=False)
            .reset_index(drop=True))
    resultados_mc[anio] = df

    # Reporte
    print(f"\n  {'Equipo':<22} {'Elo':>5} {'Oct':>6} {'QF':>6} "
          f"{'SF':>6} {'F':>6} {'🏆':>6}  Real")
    print(f"  {'-'*65}")
    for _, row in df.iterrows():
        real = ""
        if row["campeon_real"]:       real = "🏆 CAMPEÓN"
        elif row["llego_final"]:      real = "🥈 Final"
        elif row["llego_semis"]:      real = "4° Semi"
        elif row["llego_cuartos"]:    real = "QF"
        elif row["llego_octavos"]:    real = "R16"
        print(f"  {row['equipo']:<22} {row['elo']:>5} "
              f"{row['p_octavos']:>5.1f}% {row['p_cuartos']:>5.1f}% "
              f"{row['p_semis']:>5.1f}% {row['p_finalista']:>5.1f}% "
              f"{row['p_campeon']:>5.1f}%  {real}")

    # Métricas de torneo
    camp_real = reales["campeon"]
    p_camp    = df.loc[df["equipo"]==camp_real,"p_campeon"].values[0]
    semis_real = reales["semifinalistas"]
    p_semis   = df[df["equipo"].isin(semis_real)]["p_semis"].mean()
    oct_real  = reales["octavos"]
    top16_mod = set(df.nlargest(16,"p_octavos")["equipo"])
    aciertos  = len(top16_mod & oct_real)

    print(f"\n  --- Métricas de torneo ---")
    print(f"  P(campeón real — {camp_real}): {p_camp:.1f}%")
    print(f"  P(semis) promedio de los 4 semifinalistas reales: {p_semis:.1f}%")
    print(f"  Clasificados a octavos acertados (top-16 modelo): {aciertos}/16")

    # Guardar
    fn = f"mc{anio}_probabilidades.csv"
    df.to_csv(f"{OUT}/{fn}", index=False)
    kb = os.path.getsize(f"{OUT}/{fn}")/1024
    print(f"\n  Guardado: {fn} ({kb:.1f} KB)")

# Referencias globales
df_mc2018 = resultados_mc[2018]
df_mc2022 = resultados_mc[2022]

print(f"\n{'='*58}")
print(f"  RESUMEN COMPARATIVO")
print(f"{'='*58}")
for anio in [2018,2022]:
    df   = resultados_mc[anio]
    camp = REALES[anio]["campeon"]
    p    = df.loc[df["equipo"]==camp,"p_campeon"].values[0]
    rank = df[df["equipo"]==camp].index[0]+1
    print(f"  {anio}: campeón real={camp} | "
          f"P(campeón)={p:.1f}% | rank #{rank} de {len(df)}")

print(f"\n✓ BLOQUE completo")
print(f"  Variables en sesión: df_mc2018, df_mc2022")


  MONTE CARLO 2018 — 10,000 simulaciones
  Partidos neutrales en fit: 1,170
  β=0.001855 | exp(α)=1.157

  Equipo                   Elo    Oct     QF     SF      F      🏆  Real
  -----------------------------------------------------------------
  Brazil                  2130  93.5%  68.0%  44.9%  32.6%  21.8%  QF
  Germany                 2091  90.9%  61.2%  35.7%  24.9%  15.4%  
  Spain                   2048  83.8%  66.5%  42.6%  26.5%  13.9%  R16
  Portugal                2000  77.5%  57.5%  31.3%  17.6%   8.4%  R16
  France                  1994  77.1%  48.7%  29.7%  15.1%   7.3%  🏆 CAMPEÓN
  Argentina               1983  82.2%  47.7%  28.8%  14.8%   6.4%  R16
  England                 1940  81.8%  50.5%  28.1%  10.6%   5.1%  4° Semi
  Belgium                 1929  80.9%  48.3%  26.4%   9.8%   4.4%  4° Semi
  Colombia                1927  77.7%  42.8%  24.2%   8.9%   3.8%  R16
  Uruguay                 1876  85.9%  35.7%  15.3%   7.0%   2.5%  QF
  Peru                    1906  58.

In [ ]:
# =====================================================================
# BLOQUE — VALIDACIÓN ESTADÍSTICA FORMAL
# Inputs : df_validacion, df_mc2018, df_mc2022
# Outputs: tabla de métricas formales
# Guarda : validacion_estadistica.csv
# Pruebas:
#   1. Test de permutaciones (RPS vs referencia ingenua)
#   2. Accuracy de resultado (resultado más probable vs real)
#   3. Correlación Spearman (Elo pre-torneo vs ronda alcanzada)
# =====================================================================
import numpy as np, pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

for v in ["df_validacion","df_mc2018","df_mc2022","OUT"]:
    assert v in dir(), f"Falta '{v}': re-corre bloque correspondiente"

N_PERM = 10_000

# --- Funciones base --------------------------------------------------
def rps_score(probs, res):
    cp = np.cumsum(probs); cr = np.cumsum(res)
    return sum((a-b)**2 for a,b in zip(cp[:-1],cr[:-1]))/(len(probs)-1)

def res_vector(resultado):
    if resultado=="H": return [1,0,0]
    elif resultado=="D": return [0,1,0]
    else: return [0,0,1]

def resultado_mas_probable(p_home, p_draw, p_away):
    probs = {"H":p_home, "D":p_draw, "A":p_away}
    return max(probs, key=probs.get)

# --- Mapeo de ronda a número ----------------------------------------
RONDA_NUM = {
    2018: {"":0, "R16":1, "QF":2, "4° Semi":3, "🥈 Final":4, "🏆 CAMPEÓN":5},
    2022: {"":0, "R16":1, "QF":2, "4° Semi":3, "🥈 Final":4, "🏆 CAMPEÓN":5},
}

def ronda_a_numero(df_mc, anio):
    """Convierte columnas booleanas de ronda a número."""
    def ronda(row):
        if row["campeon_real"]:    return 5
        elif row["llego_final"]:   return 4
        elif row["llego_semis"]:   return 3
        elif row["llego_cuartos"]: return 2
        elif row["llego_octavos"]: return 1
        else:                      return 0
    df = df_mc.copy()
    df["ronda_real"] = df.apply(ronda, axis=1)
    return df

registros = []

for anio in [2018, 2022]:
    print(f"\n{'='*55}")
    print(f"  VALIDACIÓN ESTADÍSTICA {anio}")
    print(f"{'='*55}")

    sub = df_validacion[df_validacion["torneo"]==anio].copy()
    n   = len(sub)

    # ----------------------------------------------------------------
    # PRUEBA 1 — Test de permutaciones sobre RPS
    # ----------------------------------------------------------------
    # RPS real del modelo
    rps_modelo = []
    rps_ref    = []
    for _, row in sub.iterrows():
        probs = [row["p_home"], row["p_draw"], row["p_away"]]
        res   = res_vector(row["resultado"])
        rps_modelo.append(rps_score(probs, res))
        rps_ref.append(rps_score([1/3,1/3,1/3], res))

    rps_obs     = np.mean(rps_modelo)
    rps_ref_obs = np.mean(rps_ref)
    diferencia_obs = rps_ref_obs - rps_obs   # positivo = modelo mejor

    # Permutaciones: mezclar predicciones y resultados
    resultados_arr = sub["resultado"].values
    p_home_arr     = sub["p_home"].values
    p_draw_arr     = sub["p_draw"].values
    p_away_arr     = sub["p_away"].values

    difs_perm = []
    for _ in range(N_PERM):
        idx_perm  = np.random.permutation(n)
        rps_perm  = []
        rps_ref_p = []
        for i, j in enumerate(idx_perm):
            probs = [p_home_arr[j], p_draw_arr[j], p_away_arr[j]]
            res   = res_vector(resultados_arr[i])
            rps_perm.append(rps_score(probs, res))
            rps_ref_p.append(rps_score([1/3,1/3,1/3], res))
        difs_perm.append(np.mean(rps_ref_p) - np.mean(rps_perm))

    p_valor = np.mean(np.array(difs_perm) >= diferencia_obs)

    print(f"\n  PRUEBA 1 — Test de permutaciones (RPS, N={N_PERM:,})")
    print(f"  RPS modelo   : {rps_obs:.4f}")
    print(f"  RPS referencia: {rps_ref_obs:.4f}")
    print(f"  Diferencia observada (ref - modelo): {diferencia_obs:.4f}")
    print(f"  p-valor: {p_valor:.4f}  {'✓ significativo (p<0.05)' if p_valor<0.05 else '✗ no significativo'}")
    print(f"  Interpretación: en {p_valor*100:.1f}% de las permutaciones "
          f"el azar supera al modelo.")

    # ----------------------------------------------------------------
    # PRUEBA 2 — Accuracy de resultado
    # ----------------------------------------------------------------
    sub["pred_resultado"] = sub.apply(
        lambda r: resultado_mas_probable(r["p_home"], r["p_draw"], r["p_away"]),
        axis=1
    )
    accuracy = (sub["pred_resultado"] == sub["resultado"]).mean()

    # Distribución real de resultados
    dist_real = sub["resultado"].value_counts(normalize=True)
    # Referencia: siempre predecir el resultado más común
    ref_acc   = dist_real.max()

    # Intervalo de confianza binomial (Wilson)
    from scipy.stats import binom
    n_ok = int(accuracy * n)
    ci_low  = binom.ppf(0.025, n, accuracy) / n
    ci_high = binom.ppf(0.975, n, accuracy) / n

    print(f"\n  PRUEBA 2 — Accuracy de resultado")
    print(f"  Partidos evaluados : {n}")
    print(f"  Resultados reales  : "
          f"H={dist_real.get('H',0)*100:.1f}% "
          f"D={dist_real.get('D',0)*100:.1f}% "
          f"A={dist_real.get('A',0)*100:.1f}%")
    print(f"  Accuracy modelo    : {accuracy*100:.1f}% ({n_ok}/{n})")
    print(f"  Referencia (modo)  : {ref_acc*100:.1f}%")
    print(f"  IC 95% (Wilson)    : [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
    print(f"  {'✓ supera referencia' if accuracy > ref_acc else '✗ no supera referencia'}")

    # Desglose por tipo de resultado predicho
    print(f"\n  Desglose por resultado predicho:")
    for pred in ["H","D","A"]:
        mask = sub["pred_resultado"]==pred
        if mask.sum() > 0:
            acc_p = (sub.loc[mask,"resultado"]==pred).mean()
            print(f"    Predijo {pred}: {mask.sum():>3} partidos | "
                  f"acertó {acc_p*100:.1f}%")

    # ----------------------------------------------------------------
    # PRUEBA 3 — Correlación Spearman Elo vs ronda alcanzada
    # ----------------------------------------------------------------
    df_mc = df_mc2018 if anio==2018 else df_mc2022
    df_mc = ronda_a_numero(df_mc, anio)

    corr, p_corr = stats.spearmanr(df_mc["elo"], df_mc["ronda_real"])

    print(f"\n  PRUEBA 3 — Correlación Spearman (Elo vs ronda alcanzada)")
    print(f"  Equipos: {len(df_mc)}")
    print(f"  ρ (Spearman) = {corr:.3f}")
    print(f"  p-valor      = {p_corr:.4f}  "
          f"{'✓ significativo (p<0.05)' if p_corr<0.05 else '✗ no significativo'}")
    print(f"  Interpretación: {'correlación positiva significativa — ' if corr>0 and p_corr<0.05 else ''}"
          f"equipos con más Elo {'tendieron a llegar más lejos' if corr>0 else 'no llegaron más lejos'}.")

    registros.append({
        "torneo":          anio,
        "n_partidos":      n,
        "rps_modelo":      round(rps_obs,4),
        "rps_referencia":  round(rps_ref_obs,4),
        "perm_p_valor":    round(p_valor,4),
        "perm_sig":        p_valor < 0.05,
        "accuracy":        round(accuracy,4),
        "accuracy_ref":    round(ref_acc,4),
        "acc_supera_ref":  accuracy > ref_acc,
        "spearman_rho":    round(corr,3),
        "spearman_p":      round(p_corr,4),
        "spearman_sig":    p_corr < 0.05,
    })

# --- Resumen conjunto ------------------------------------------------
df_val_est = pd.DataFrame(registros)

print(f"\n{'='*55}")
print(f"  RESUMEN ESTADÍSTICO CONJUNTO")
print(f"{'='*55}")
print(f"\n  {'Prueba':<35} {'2018':>10} {'2022':>10}")
print(f"  {'-'*55}")
print(f"  {'RPS modelo':<35} "
      f"{df_val_est.loc[0,'rps_modelo']:>10.4f} "
      f"{df_val_est.loc[1,'rps_modelo']:>10.4f}")
print(f"  {'p-valor permutaciones':<35} "
      f"{df_val_est.loc[0,'perm_p_valor']:>10.4f} "
      f"{df_val_est.loc[1,'perm_p_valor']:>10.4f}")
print(f"  {'Accuracy':<35} "
      f"{df_val_est.loc[0,'accuracy']*100:>9.1f}% "
      f"{df_val_est.loc[1,'accuracy']*100:>9.1f}%")
print(f"  {'Accuracy referencia':<35} "
      f"{df_val_est.loc[0,'accuracy_ref']*100:>9.1f}% "
      f"{df_val_est.loc[1,'accuracy_ref']*100:>9.1f}%")
print(f"  {'Spearman ρ (Elo vs ronda)':<35} "
      f"{df_val_est.loc[0,'spearman_rho']:>10.3f} "
      f"{df_val_est.loc[1,'spearman_rho']:>10.3f}")
print(f"  {'Spearman p-valor':<35} "
      f"{df_val_est.loc[0,'spearman_p']:>10.4f} "
      f"{df_val_est.loc[1,'spearman_p']:>10.4f}")

# --- Guardar --------------------------------------------------------
df_val_est.to_csv(f"{OUT}/validacion_estadistica.csv", index=False)
kb = os.path.getsize(f"{OUT}/validacion_estadistica.csv")/1024
print(f"\nGuardado: validacion_estadistica.csv ({kb:.1f} KB)")
print(f"\n✓ BLOQUE validación estadística completo")


  VALIDACIÓN ESTADÍSTICA 2018

  PRUEBA 1 — Test de permutaciones (RPS, N=10,000)
  RPS modelo   : 0.2080
  RPS referencia: 0.2439
  Diferencia observada (ref - modelo): 0.0359
  p-valor: 0.0003  ✓ significativo (p<0.05)
  Interpretación: en 0.0% de las permutaciones el azar supera al modelo.

  PRUEBA 2 — Accuracy de resultado
  Partidos evaluados : 64
  Resultados reales  : H=39.1% D=20.3% A=40.6%
  Accuracy modelo    : 54.7% (35/64)
  Referencia (modo)  : 40.6%
  IC 95% (Wilson)    : [42.2%, 67.2%]
  ✓ supera referencia

  Desglose por resultado predicho:
    Predijo H:  33 partidos | acertó 54.5%
    Predijo A:  31 partidos | acertó 54.8%

  PRUEBA 3 — Correlación Spearman (Elo vs ronda alcanzada)
  Equipos: 32
  ρ (Spearman) = 0.538
  p-valor      = 0.0015  ✓ significativo (p<0.05)
  Interpretación: correlación positiva significativa — equipos con más Elo tendieron a llegar más lejos.

  VALIDACIÓN ESTADÍSTICA 2022

  PRUEBA 1 — Test de permutaciones (RPS, N=10,000)
  RPS modelo 

In [ ]:
from scipy.stats import poisson
import numpy as np, pandas as pd

r = elo_pre.set_index("country")["rating"]

def predecir(eq_h, eq_a):
    # Verificar que ambos equipos están en los 48
    for eq in [eq_h, eq_a]:
        if eq not in r.index:
            similares = [e for e in r.index if eq.lower() in e.lower()]
            print(f"✗ '{eq}' no encontrado.")
            if similares: print(f"  ¿Quisiste decir: {similares}?")
            return None

    # Parámetros
    eh, ea = float(r[eq_h]), float(r[eq_a])
    lh = np.exp(a_n + b_n*(eh-ea))
    la = np.exp(a_n + b_n*(ea-eh))

    # Matriz de marcadores (hasta 6 goles por equipo)
    MAXG = 6
    rng  = np.arange(MAXG+1)
    M    = np.outer(poisson.pmf(rng, lh), poisson.pmf(rng, la))

    # Probabilidades de resultado
    p_home = float(np.tril(M,-1).sum())
    p_draw = float(np.trace(M))
    p_away = float(np.triu(M,1).sum())

    # Top 10 marcadores más probables
    marcadores = []
    for i in range(MAXG+1):
        for j in range(MAXG+1):
            marcadores.append({
                "marcador": f"{i}-{j}",
                "prob":     round(float(M[i,j])*100, 2)
            })
    df_marc = (pd.DataFrame(marcadores)
                 .sort_values("prob", ascending=False)
                 .head(10)
                 .reset_index(drop=True))
    df_marc.index += 1

    # Imprimir reporte
    print(f"\n{'='*45}")
    print(f"  {eq_h}  vs  {eq_a}")
    print(f"  Elo: {eh:.0f}  vs  {ea:.0f}")
    print(f"{'='*45}")
    print(f"\n  Goles esperados : {eq_h} {lh:.2f} — {la:.2f} {eq_a}")
    print(f"\n  {'Resultado':<12} {'Probabilidad':>12}")
    print(f"  {'-'*25}")
    print(f"  {eq_h+' gana':<12} {p_home*100:>11.1f}%")
    print(f"  {'Empate':<12} {p_draw*100:>11.1f}%")
    print(f"  {eq_a+' gana':<12} {p_away*100:>11.1f}%")
    print(f"\n  Top 10 marcadores más probables:")
    print(f"  {'#':<4} {'Marcador':<12} {'Prob':>8}")
    print(f"  {'-'*25}")
    for idx, row in df_marc.iterrows():
        print(f"  {idx:<4} {row['marcador']:<12} {row['prob']:>7.2f}%")

    return {
        "eq_h": eq_h, "eq_a": eq_a,
        "elo_h": eh, "elo_a": ea,
        "lambda_h": round(lh,3), "lambda_a": round(la,3),
        "p_home": round(p_home,4),
        "p_draw": round(p_draw,4),
        "p_away": round(p_away,4),
        "marcadores": df_marc
    }

In [ ]:
# =====================================================================
# GENERAR TODAS LAS PREDICCIONES POSIBLES (48×47 combinaciones)
# Guarda : predicciones_todos_partidos.csv
# =====================================================================
import numpy as np, pandas as pd
from scipy.stats import poisson

r = elo_pre.set_index("country")["rating"]
equipos = sorted(r.index.tolist())

registros = []
MAXG = 6
rng  = np.arange(MAXG+1)

for eq_h in equipos:
    for eq_a in equipos:
        if eq_h == eq_a:
            continue
        eh, ea = float(r[eq_h]), float(r[eq_a])
        lh = np.exp(a_n + b_n*(eh-ea))
        la = np.exp(a_n + b_n*(ea-eh))
        M  = np.outer(poisson.pmf(rng, lh), poisson.pmf(rng, la))

        p_home = float(np.tril(M,-1).sum())
        p_draw = float(np.trace(M))
        p_away = float(np.triu(M,1).sum())

        # Marcador más probable
        idx    = np.unravel_index(M.argmax(), M.shape)
        marc_mp = f"{idx[0]}-{idx[1]}"
        prob_mp = float(M[idx])

        registros.append({
            "equipo_h":      eq_h,
            "equipo_a":      eq_a,
            "elo_h":         int(eh),
            "elo_a":         int(ea),
            "lambda_h":      round(lh, 3),
            "lambda_a":      round(la, 3),
            "p_victoria_h":  round(p_home*100, 1),
            "p_empate":      round(p_draw*100, 1),
            "p_victoria_a":  round(p_away*100, 1),
            "marcador_mp":   marc_mp,
            "prob_mp":       round(prob_mp*100, 2),
        })

df_pred = pd.DataFrame(registros)
df_pred.to_csv(f"{OUT}/predicciones_todos_partidos.csv", index=False)
kb = os.path.getsize(f"{OUT}/predicciones_todos_partidos.csv")/1024

print(f"Combinaciones generadas: {len(df_pred):,}")
print(f"Guardado: predicciones_todos_partidos.csv ({kb:.1f} KB)")
print(f"\nEjemplo — Mexico vs South Africa:")
ej = df_pred[(df_pred["equipo_h"]=="Mexico") &
             (df_pred["equipo_a"]=="South Africa")].iloc[0]
print(f"  λ={ej['lambda_h']:.2f}-{ej['lambda_a']:.2f} | "
      f"W={ej['p_victoria_h']}% D={ej['p_empate']}% L={ej['p_victoria_a']}% | "
      f"Marcador MP: {ej['marcador_mp']} ({ej['prob_mp']}%)")

Combinaciones generadas: 2,256
Guardado: predicciones_todos_partidos.csv (142.5 KB)

Ejemplo — Mexico vs South Africa:
  λ=1.74-0.94 | W=56.1% D=23.5% L=20.1% | Marcador MP: 1-0 (11.91%)


In [ ]:
# =====================================================================
# BLOQUE — VISUALIZACIONES DEL README
# Inputs : df_resultados, df_validacion, df_mc2018, df_mc2022, OUT
# Outputs: 3 archivos PNG
# Guarda : figures/top10_favoritos_2026.png
#          figures/validacion_metricas.png
#          figures/calibracion_elo_ronda.png
# Todos los números se leen de las variables reales en sesión,
# nada hardcodeado.
# =====================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

for v in ["df_resultados","df_validacion","df_mc2018","df_mc2022","OUT"]:
    assert v in dir(), f"Falta '{v}': re-corre el bloque correspondiente"

FIG_DIR = f"{OUT}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams['font.family'] = 'DejaVu Sans'

# =====================================================================
# GRÁFICO 1 — Top 10 favoritos al título, Mundial 2026
# Fuente: df_resultados (columna p_campeon)
# =====================================================================
top10 = df_resultados.nlargest(10, "p_campeon").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.Blues(np.linspace(0.9, 0.4, len(top10)))
bars = ax.barh(top10["equipo"][::-1], top10["p_campeon"][::-1],
                color=colors[::-1], edgecolor='white', height=0.65)

for bar, p, elo in zip(bars, top10["p_campeon"][::-1], top10["elo"][::-1]):
    ax.text(p + 0.15, bar.get_y()+bar.get_height()/2, f"{p:.1f}%",
            va='center', fontsize=10, fontweight='bold', color='#1a1a1a')
    ax.text(0.15, bar.get_y()+bar.get_height()/2, f"Elo {elo}",
            va='center', fontsize=8, color='white', fontweight='medium')

ax.set_xlabel("Probabilidad de campeonato (%)", fontsize=10)
ax.set_title("Top 10 favoritos al título — Mundial 2026\n"
              "10,000 simulaciones Monte Carlo · Modelo Elo + Poisson",
              fontsize=12, fontweight='bold', pad=15)
ax.set_xlim(0, top10["p_campeon"].max()*1.15)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top10_favoritos_2026.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ top10_favoritos_2026.png")

# =====================================================================
# GRÁFICO 2 — Validación: Modelo vs Referencia ingenua
# Fuente: df_validacion (agregado por torneo)
# =====================================================================
def metricas_torneo(df, anio):
    sub = df[df["torneo"]==anio]
    modelo = [sub["brier"].mean(), sub["logloss"].mean(), sub["rps"].mean()]
    # Referencia: recalculada con las mismas funciones, prob=1/3 fijo
    ref = []
    for _, row in sub.iterrows():
        res = ([1,0,0] if row["resultado"]=="H"
                else [0,1,0] if row["resultado"]=="D" else [0,0,1])
        pass
    # Referencia ingenua exacta (constante dado res, se recalcula directo):
    from itertools import product
    def brier(p,r): return sum((a-b)**2 for a,b in zip(p,r))
    def logloss(p,r):
        eps=1e-10; return -sum(rr*np.log(pp+eps) for pp,rr in zip(p,r))
    def rps(p,r):
        cp=np.cumsum(p); cr=np.cumsum(r)
        return sum((a-b)**2 for a,b in zip(cp[:-1],cr[:-1]))/(len(p)-1)
    REF=[1/3,1/3,1/3]
    bs,ll,rp = [],[],[]
    for _, row in sub.iterrows():
        res = ([1,0,0] if row["resultado"]=="H"
                else [0,1,0] if row["resultado"]=="D" else [0,0,1])
        bs.append(brier(REF,res)); ll.append(logloss(REF,res)); rp.append(rps(REF,res))
    ref = [np.mean(bs), np.mean(ll), np.mean(rp)]
    return modelo, ref

metricas = ['Brier Score','Log Loss','RPS']
modelo_2018, ref_2018 = metricas_torneo(df_validacion, 2018)
modelo_2022, ref_2022 = metricas_torneo(df_validacion, 2022)

fig, axes = plt.subplots(1, 2, figsize=(11,5))
for ax, modelo, ref, anio in zip(axes, [modelo_2018,modelo_2022],
                                          [ref_2018,ref_2022], [2018,2022]):
    x = np.arange(len(metricas)); w = 0.32
    b1 = ax.bar(x-w/2, modelo, w, label='Modelo', color='#2563eb', edgecolor='white')
    b2 = ax.bar(x+w/2, ref, w, label='Referencia ingenua', color='#cbd5e1', edgecolor='white')
    for bars in [b1,b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.012, f"{h:.3f}",
                    ha='center', fontsize=8.5, color='#333')
    ax.set_xticks(x); ax.set_xticklabels(metricas, fontsize=10)
    ax.set_title(f"Mundial {anio}", fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(ref)*1.3)
    ax.spines[['top','right']].set_visible(False)
    ax.legend(fontsize=8.5, loc='upper right', frameon=False)

fig.suptitle("Validación: modelo vs. referencia aleatoria (más bajo = mejor)",
              fontsize=12.5, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/validacion_metricas.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ validacion_metricas.png")

# =====================================================================
# GRÁFICO 3 — Correlación Elo vs Ronda alcanzada (2018 + 2022)
# Fuente: df_mc2018, df_mc2022 (columnas elo + flags llego_*)
# Spearman se recalcula aquí, no se hardcodea.
# =====================================================================
def ronda_numero(row):
    if row["campeon_real"]:    return 5
    elif row["llego_final"]:   return 4
    elif row["llego_semis"]:   return 3
    elif row["llego_cuartos"]: return 2
    elif row["llego_octavos"]: return 1
    else:                      return 0

df18 = df_mc2018.copy(); df18["ronda"] = df18.apply(ronda_numero, axis=1)
df22 = df_mc2022.copy(); df22["ronda"] = df22.apply(ronda_numero, axis=1)

rho18, p18 = stats.spearmanr(df18["elo"], df18["ronda"])
rho22, p22 = stats.spearmanr(df22["elo"], df22["ronda"])

camp18 = df18[df18["campeon_real"]].iloc[0]
camp22 = df22[df22["campeon_real"]].iloc[0]

np.random.seed(42)
jitter = lambda v: np.array(v) + np.random.uniform(-0.12, 0.12, len(v))

fig, ax = plt.subplots(figsize=(8.5,6))
ax.scatter(df18["elo"], jitter(df18["ronda"]), s=70, alpha=0.75, color='#f97316',
           edgecolor='white', linewidth=0.8, label=f'2018 (ρ={rho18:.2f}, p={p18:.4f})')
ax.scatter(df22["elo"], jitter(df22["ronda"]), s=70, alpha=0.75, color='#2563eb',
           edgecolor='white', linewidth=0.8, label=f'2022 (ρ={rho22:.2f}, p={p22:.4f})')

all_elo   = np.concatenate([df18["elo"], df22["elo"]])
all_ronda = np.concatenate([df18["ronda"], df22["ronda"]])
z = np.polyfit(all_elo, all_ronda, 1)
xline = np.linspace(all_elo.min()-20, all_elo.max()+20, 100)
ax.plot(xline, np.polyval(z, xline), '--', color='#475569', linewidth=1.6,
        alpha=0.8, label='Tendencia conjunta')

ax.annotate(f'{camp18["equipo"]} (campeón 2018)', (camp18["elo"], 5),
            xytext=(camp18["elo"]-140, 4.55), fontsize=8.5, color='#f97316',
            fontweight='bold', arrowprops=dict(arrowstyle='->', color='#f97316', lw=1))
ax.annotate(f'{camp22["equipo"]} (campeón 2022)', (camp22["elo"], 5),
            xytext=(camp22["elo"]-170, 4.7), fontsize=8.5, color='#2563eb',
            fontweight='bold', arrowprops=dict(arrowstyle='->', color='#2563eb', lw=1))

ax.set_yticks([0,1,2,3,4,5])
ax.set_yticklabels(['Grupos','R16','Cuartos','Semis','Final','Campeón'], fontsize=9.5)
ax.set_xlabel("Elo pre-torneo", fontsize=10.5)
ax.set_title("¿El Elo predice qué tan lejos llega un equipo?\n"
              "Mundiales 2018 y 2022 — correlación de Spearman",
              fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=9, loc='upper left', frameon=False)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/calibracion_elo_ronda.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ calibracion_elo_ronda.png")

# --- Verificación final -----------------------------------------------
print(f"\nGuardado en: {FIG_DIR}")
for fn in ["top10_favoritos_2026.png","validacion_metricas.png","calibracion_elo_ronda.png"]:
    kb = os.path.getsize(f"{FIG_DIR}/{fn}")/1024
    print(f"  {fn:<35} {kb:7.1f} KB")

print(f"\nSpearman recalculado (debe coincidir con validación estadística previa):")
print(f"  2018: ρ={rho18:.3f}, p={p18:.4f}")
print(f"  2022: ρ={rho22:.3f}, p={p22:.4f}")

✓ top10_favoritos_2026.png
✓ validacion_metricas.png
✓ calibracion_elo_ronda.png

Guardado en: /content/drive/MyDrive/wc2026_forecasting/v2/figures
  top10_favoritos_2026.png               83.7 KB
  validacion_metricas.png                53.1 KB
  calibracion_elo_ronda.png              96.4 KB

Spearman recalculado (debe coincidir con validación estadística previa):
  2018: ρ=0.538, p=0.0015
  2022: ρ=0.459, p=0.0083
